# Assemble + publish chessbench-full

Downloads the 8 shard pieces from HF, merges them into `train_set.npz` + `teacher_logp.npy` (runbook contract), and publishes the Kaggle Dataset `chessbench-full` (public, so all three accounts can mount it). Run once after all shards are built.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO = Path('/kaggle/working/chess-slm-benchmark')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Vedang-P/chess-slm-benchmark.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_WRITE_TOKEN'] = UserSecretsClient().get_secret('HF_WRITE_TOKEN')
except Exception as exc:
    print('HF secret unavailable:', exc)
os.chdir(REPO)

In [ ]:
# Self-wait: poll HF until all 8 shards are built (builders run in parallel
# across accounts; this CPU kernel waits and assembles automatically).
import json, time
from huggingface_hub import hf_hub_download
import os
from pathlib import Path
while True:
    try:
        m = hf_hub_download('vedangfake/chess-slm-benchmark', 'chessbench-full-build/manifest.json',
                           repo_type='dataset', token=os.environ.get('HF_WRITE_TOKEN'))
        man = json.loads(Path(m).read_text())
        n_done = len(man.get('shards', {}))
        print(f'[wait] shards ready: {n_done}/8', flush=True)
        if n_done >= 8:
            break
    except Exception as exc:
        print(f'[wait] manifest not ready: {exc}', flush=True)
    time.sleep(300)

In [ ]:
cmd = [sys.executable, 'scripts/assemble_full_dataset.py',
       '--n-shards', '8',
       '--hf-repo', 'vedangfake/chess-slm-benchmark',
       '--hf-run', 'chessbench-full-build',
       '--out', '/kaggle/working/chessbench-full']
print(' '.join(cmd))
subprocess.run(cmd, check=True)
print('ASSEMBLED')

In [ ]:
# Publish as a public Kaggle Dataset so all three accounts can mount it.
import json
from pathlib import Path
out = Path('/kaggle/working/chessbench-full')
meta = {
    'id': 'ACCOUNT/chessbench-full',
    'title': 'chessbench-full',
    'subtitle': 'ChessBench train action-value slice: tokens/actions/winprob + 9M teacher log-probs [8 shards]',
    'isPrivate': False,
    'licenses': [{'name': 'other'}]}
(out / 'dataset-metadata.json').write_text(json.dumps(meta, indent=2))
import os
os.environ['KAGGLE_USERNAME'] = 'ACCOUNT'
r = subprocess.run(['kaggle', 'datasets', 'create', '-p', str(out)],
                   capture_output=True, text=True)
print(r.stdout[-2000:])
print(r.stderr[-2000:])
print('DATASET CREATED' if r.returncode == 0 else 'PUBLISH FAILED - see output')